#1.Semilla y características del split

En la variable **SEED** se fija la semilla de la cual depende todo el notebook.

Para la división en train-val-test seguimos el siguiente criterio:
* El porcentaje de test y el estratificado se fijan en función de las reglas del reto.
* Como mínimo usamos un 60% de los datos para el conjunto de train.
* Los datos que caen fuera de test se dividen en train y val respetando siempre la condición anterior.

El conjunto de train se usará para entrenar los modelos, el de validación para ajustar la calibración a partir de PD, EAD y LGD predichas sobre este conjunto y el test solo se utiliza para obtener los resultados finales sobre un conjunto de datos que no ha sido utilizado en ninguna de las fases de modelización y calibración.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42                 #Semilla de la cual depende todo el notebook

# --- Porcentajes ---
PERC_TEST = 0.20            # porcentaje de test (el oficial del reto)
MIN_TRAIN = 0.60            # train nunca por debajo del 60 % del total
# Validación = mismo tamaño que el test (es la muestra de calibración y simula el test),
# salvo que eso deje train por debajo de MIN_TRAIN: entonces se reduce lo necesario.
PERC_VAL = min(PERC_TEST, 1 - MIN_TRAIN - PERC_TEST)
PERC_TRAIN = 1 - PERC_VAL - PERC_TEST
assert PERC_VAL > 0, "PERC_TEST demasiado grande: no queda sitio para validación con train ≥ MIN_TRAIN"
STRATIFY = False             # partición no estratificada

np.random.seed(SEED)

# 2.Procesamiento

En primer lugar se cargan los datos y se realiza la división en train-val-test.

In [2]:
# =====================================================================
# CONFIGURACIÓN Y CARGA DE DATOS
# =====================================================================

# --- Carga de datos (Colab + Google Drive) ---
from google.colab import drive
drive.mount("/content/drive")
RUTA_DATOS = "/content/drive/MyDrive/Proyectos Data Science/"
X = pd.read_csv(f"{RUTA_DATOS}X.csv", header=0)
y = pd.read_csv(f"{RUTA_DATOS}target.csv").squeeze().rename("y").astype(int)
assert len(X) == len(y) and X.index.equals(y.index)

def split_train_val_test(X, y, perc_val, perc_test, seed, stratify=True):
    """Divide en train / val / test. El test depende solo de seed y perc_test (no de perc_val).
    Devuelve copias independientes, con el índice original de cada préstamo."""
    # 1) Test
    X_resto, X_test, y_resto, y_test = train_test_split(
        X, y, test_size=perc_test, random_state=seed, stratify=y if stratify else None)
    # 2) Validación dentro del resto: su proporción se reescala al tamaño del resto
    X_train, X_val, y_train, y_val = train_test_split(
        X_resto, y_resto, test_size=perc_val / (1 - perc_test), random_state=seed,
        stratify=y_resto if stratify else None)
    return (X_train.copy(), X_val.copy(), X_test.copy(),
            y_train.copy(), y_val.copy(), y_test.copy())

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    X, y, PERC_VAL, PERC_TEST, SEED, STRATIFY)

# --- Comprobaciones ---
assert len(set(X_train.index) & set(X_val.index)) == 0
assert len(set(X_train.index) & set(X_test.index)) == 0
assert len(set(X_val.index) & set(X_test.index)) == 0
assert len(X_train) + len(X_val) + len(X_test) == len(X)
print(f"SEED = {SEED} | stratify = {STRATIFY}")
for nombre, yy in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{nombre:5s}: {len(yy):>9,} préstamos ({len(yy) / len(y):.1%}) | tasa de impago = {yy.mean():.2%}")
del X, y      # a partir de aquí solo existen los tres conjuntos

Mounted at /content/drive
SEED = 42 | stratify = False
train: 1,283,785 préstamos (60.0%) | tasa de impago = 13.09%
val  :   427,929 préstamos (20.0%) | tasa de impago = 13.01%
test :   427,929 préstamos (20.0%) | tasa de impago = 13.08%


En segundo lugar, se realiza un preprocesamiento común a las matrices $X_{train}$, $X_{val}$ y $X_{test}$.

IMPORTANTE: Es necesario cargar el archivo **emp_title_clasification.py** que se encuentra en el repositorio.

In [3]:
# =====================================================================
# INGENIERÍA DE VARIABLES COMÚN
# Lo que se aprende de los datos (clasificador de emp_title y mediana de bal_to_cred_lim) se ajusta SOLO
# con train y se aplica a val y test. Los cortes de las _cat son constantes de diseño fijadas en la EDA.
# =====================================================================
from emp_title_clasification import ClasificadorEmpTitle

# ----- Variables de desempeño: nunca entran en la PD; solo sirven para construir EAD, LGD y pérdida -----
VARIABLES_DESEMPEÑO = ["remaining_princ_for_tot_amnt_fund", "paym_rec", "princ_rec", "interest_rec", "late_fees_rec"]

# ----- Constantes de diseño (EDA) -----
MAPA_EMP_LENGTH = {
    "< 1 year": "< 2 years", "1 year": "< 2 years", "2 years": "2 years", "3 years": "3 years",
    "4 years": "4 years", "5 years": "5-6 years", "6 years": "5-6 years", "7 years": "7 years",
    "8 years": "8 years", "9 years": "9 years", "10+ years": "10+ years", "unknown": "unknown",
}
MESES = {"Jan": 1, "Feb": 2, "Mar": 3, "Apr": 4, "May": 5, "Jun": 6,
         "Jul": 7, "Aug": 8, "Sep": 9, "Oct": 10, "Nov": 11, "Dec": 12}
VENTANA_BURO = 84
COLS_DELINQ = ["mths_since_last_delinq", "mths_since_recent_revol_delinq", "mths_since_recent_bankcard_delinq"]

# Bloque de buró con nulos estructurales (préstamos anteriores a dic-2015)
BLOQUE_BURO_COMPLETO = ["max_bal_owed", "num_rev_trades_op_in_12mths", "num_inq", "num_installment_acc_op_in_12mths",
                        "num_open_trades_in_6mths", "num_installment_acc_op_in_24mths", "num_rev_trades_op_in_24mths"]

CORTES_INSTALL = [0, 3, 6, 9, 12, 15, 18, 21, 30, 60, np.inf]
ETIQUETAS_INSTALL = ["0-3", "3-6", "6-9", "9-12", "12-15", "15-18", "18-21", "21-30", "30-60", ">60"]
CORTES_BAL = [0, 25, 50, 75, 100, np.inf]
ETIQUETAS_BAL = ["0-25", "25-50", "50-75", "75-100", ">100"]
CORTES_DEROG = [-0.1, 0, 1, 2, 5, np.inf]
ETIQUETAS_DEROG = ["0", "1", "2", "3-5", ">=6"]
CORTES_LAST_DELINQ = [0, 6, 18, 36, 60, 72, 84]
ETIQUETAS_LAST_DELINQ = ["0-6", "6-18", "18-36", "36-60", "60-72", "72-84"]

# Columnas que se eliminan al final
COLUMNAS_ELIMINADAS = [
    "emp_length", "emp_title", "emp_title_norm", "emp_tipo", "region_code",
    "mths_since_last_installment_acc_op", "bal_to_cred_lim", "annual_income", "num_inq_in_12mths",
    "monthly_payment", "earliest_cr_line_year", "earliest_cr_line_month",
    "mths_since_recent_revol_delinq", "mths_since_recent_bankcard_delinq",
    "never_revol_delinq", "never_bankcard_delinq",
    "num_installment_acc_op_in_12mths", "num_rev_trades_op_in_12mths",
]


def preprocesar(X_train, X_val, X_test):
    """Ingeniería de variables común a los tres modelos. No modifica las tablas de entrada."""
    conjuntos = {"train": X_train.copy(), "val": X_val.copy(), "test": X_test.copy()}

    # --- Lo que se aprende en train ---
    tr = conjuntos["train"]
    emp_tr = tr["emp_title"].str.strip()
    emp_tr = emp_tr.mask(emp_tr.str.lower().isin({"", "na", "test", "."})).fillna("unknown") #
    clasificador_emp = ClasificadorEmpTitle().fit(emp_tr)
    bloque_nulo_tr = tr[BLOQUE_BURO_COMPLETO].isna().all(axis=1)
    mediana_bal = tr["bal_to_cred_lim"].median()

    for nombre, df in conjuntos.items():
        # 0) Nombre corto del total cobrado
        df.rename(columns={"paym_rec_for_tot_amnt_fund": "paym_rec"}, inplace=True)

        # 1) emp_title y emp_length
        emp = df["emp_title"].str.strip()
        df["emp_title"] = emp.mask(emp.str.lower().isin({"", "na", "test", "."})).fillna("unknown")
        df["emp_length_cat"] = df["emp_length"].fillna("unknown").map(MAPA_EMP_LENGTH)
        df[["emp_title_norm", "emp_tipo", "emp_ambito"]] = clasificador_emp.transform(df["emp_title"])

        # 2) Antigüedad del historial crediticio en meses
        df["credit_history_months"] = (
            (pd.to_numeric(df["issue_date_year"], errors="coerce")
             - pd.to_numeric(df["earliest_cr_line_year"], errors="coerce")) * 12
            + (df["issue_date_month"].str.strip().map(MESES) - df["earliest_cr_line_month"].str.strip().map(MESES)))

        # 3) Morosidad: recorte a 84 meses, flag de "sin retraso registrado" e imputación
        df[COLS_DELINQ] = df[COLS_DELINQ].clip(upper=VENTANA_BURO)
        df["never_delinq"] = df["mths_since_last_delinq"].isna().astype(int)
        periodo_verificable = df["credit_history_months"].clip(lower=0, upper=VENTANA_BURO)
        for c in COLS_DELINQ:
            df[c] = df[c].fillna(periodo_verificable)

        # 4) Meses desde la última apertura de préstamo a plazos, en tramos
        bloque_nulo = df[BLOQUE_BURO_COMPLETO].isna().all(axis=1)
        v = df["mths_since_last_installment_acc_op"]
        cat = pd.cut(v, bins=CORTES_INSTALL, labels=ETIQUETAS_INSTALL, include_lowest=True).astype("object")
        cat[v.isna() & bloque_nulo] = "sin_registro"
        cat[v.isna() & ~bloque_nulo] = "nunca"
        df["mths_since_last_installment_acc_op_cat"] = cat

        # 5) Uso del límite de crédito: imputación de nulos fuera del bloque y tramos
        extra = df["bal_to_cred_lim"].isna() & ~bloque_nulo
        df.loc[extra, "bal_to_cred_lim"] = mediana_bal
        cat = pd.cut(df["bal_to_cred_lim"], bins=CORTES_BAL, labels=ETIQUETAS_BAL,
                     include_lowest=True).astype("object")
        cat[bloque_nulo] = "sin_registro"
        df["bal_to_cred_lim_cat"] = cat

        # 6) Ingresos en log10
        df["annual_income_log"] = np.log10(df["annual_income"])

        # 7) Consultas de crédito entre 7 y 12 meses
        df["num_inq_in_7_12mths"] = (df["num_inq_in_12mths"] - df["num_inq_in_6mths"]).clip(lower=0)

        # 8) Registros públicos derogatorios en tramos
        df["num_derogatory_pub_rec_cat"] = pd.cut(df["num_derogatory_pub_rec"], bins=CORTES_DEROG,
                                                  labels=ETIQUETAS_DEROG).astype("object")

        # 9) Meses desde el último retraso en tramos
        cat = pd.cut(df["mths_since_last_delinq"], bins=CORTES_LAST_DELINQ, labels=ETIQUETAS_LAST_DELINQ,
                     include_lowest=True).astype("object")
        cat[df["never_delinq"] == 1] = "sin retraso registrado"
        df["mths_since_last_delinq_cat"] = cat

        # 10) Eliminación de las columnas sustituidas o descartadas
        df.drop(columns=[c for c in COLUMNAS_ELIMINADAS if c in df.columns], inplace=True)

    # --- Comprobaciones ---
    nuevas = ["emp_length_cat", "emp_ambito", "credit_history_months", "never_delinq",
              "mths_since_last_installment_acc_op_cat", "bal_to_cred_lim_cat", "annual_income_log",
              "num_inq_in_7_12mths", "num_derogatory_pub_rec_cat", "mths_since_last_delinq_cat"]
    for nombre, df in conjuntos.items():
        nulos = df[nuevas].isna().sum()
        assert list(df.columns) == list(conjuntos["train"].columns), f"{nombre}: columnas distintas a train"
        print(f"{nombre:5s}: {df.shape} | nulos en variables creadas: {nulos[nulos > 0].to_dict() or 0}")

    return conjuntos["train"], conjuntos["val"], conjuntos["test"]


X_train, X_val, X_test = preprocesar(X_train, X_val, X_test)

train: (1283785, 41) | nulos en variables creadas: 0
val  : (427929, 41) | nulos en variables creadas: 0
test : (427929, 41) | nulos en variables creadas: 0


A continuación, se crean las matrices que se utilizarán en cada uno de los modelos.

Para el modelo de PD se sustituyen las variables categóricas por sus correspondientes valores de WoE. Se añade la opción `WOE_FUERA_DE_MUESTRA` que permite realizar el cálculo del WoE mediante  un KFold para que ninguno de los préstamos aprenda su valor correspondiente de WoE usando su propia target, evitando así generar predicciones demasiado optimistas al usar estas variables como predictoras.

En la versión final del proyecto, la optimización de hiperparámetros mediante Optuna del modelo de PD se realizó con `WOE_FUERA_DE_MUESTRA`= True, por tanto, activaremos esta opción.

In [4]:
# =====================================================================
# MATRICES DE CADA MODELO Y TARGETS
#   PD : numéricas estandarizadas + buro_sin_registro + WoE de 8 categóricas
#   EAD: impagos de train; cartera completa de val y test
#   LGD: impagos de train con EAD > 0; cartera completa de val y test
# Todo lo que se aprende (WoE, medianas, medias, desviaciones, categorías) sale SOLO de train.
# =====================================================================

# ----------------------------- PD -----------------------------
from sklearn.model_selection import KFold

NUM_PD = ["interest_rate", "issue_date_year", "num_rev_trades_op_in_24mths", "num_open_trades_in_6mths",
          "num_installment_acc_op_in_24mths", "max_bal_owed", "num_inq", "loan_term_months",
          "dept_paym_income_ratio", "used_credit_share", "num_inq_in_6mths", "annual_income_log", "funded_amnt"]
CAT_PD = ["bal_to_cred_lim_cat", "mths_since_last_installment_acc_op_cat", "verification_status",
          "disbursement_method", "initial_list_status", "loan_purpose", "emp_ambito", "home_ownership_status"]
BLOQUE_BURO = ["num_open_trades_in_6mths", "num_installment_acc_op_in_24mths",
               "num_rev_trades_op_in_24mths", "max_bal_owed", "num_inq"]
MIN_N_WOE = 500                 # categorías con menos préstamos se agrupan en "otros"
SUAVIZADO_WOE = 0.5
WOE_FUERA_DE_MUESTRA = True    # True: WoE de train con KFold
N_BLOQUES_WOE = 5

def tabla_woe(x, y):
    """WoE por categoría. Nulos -> "nulo"; categorías con menos de MIN_N_WOE préstamos -> "otros"."""
    t = x.astype(str)
    frec = t.value_counts()
    t = t.where(t.isin(frec[frec >= MIN_N_WOE].index), "otros").where(x.notna(), "nulo")
    c = pd.crosstab(t, y)
    malos, buenos = c[1] + SUAVIZADO_WOE, c[0] + SUAVIZADO_WOE
    return np.log((buenos / buenos.sum()) / (malos / malos.sum())).rename("woe")

def aplicar_woe(x, woe):
    """Categoría no vista o pequeña -> "otros"; si "otros" no existe en la tabla -> WoE 0 (riesgo medio)."""
    v = x.astype(str).where(x.notna(), "nulo")
    return v.where(v.isin(woe.index), "otros").map(woe).fillna(0).astype("float32")

def woe_train_fuera_de_muestra(X_train, y_train, seed):
    """WoE de train calculado fuera de muestra: cada préstamo recibe el WoE de su categoría estimado
    con los otros bloques, de modo que su propio impago no interviene en su valor."""
    woe_tr = pd.DataFrame(index=X_train.index, columns=[f"{c}_woe" for c in CAT_PD], dtype="float32")
    for i_aj, i_ap in KFold(N_BLOQUES_WOE, shuffle=True, random_state=seed).split(X_train):
        i_aj, i_ap = X_train.index[i_aj], X_train.index[i_ap]
        for c in CAT_PD:
            woe = tabla_woe(X_train.loc[i_aj, c], y_train.loc[i_aj])
            woe_tr.loc[i_ap, f"{c}_woe"] = aplicar_woe(X_train.loc[i_ap, c], woe).to_numpy()
    return woe_tr.astype("float32")

def matrices_pd(X_train, X_val, X_test, y_train, seed):
    woes = {c: tabla_woe(X_train[c], y_train) for c in CAT_PD}     # con todo train: para val y test
    bloque_pd = [c for c in BLOQUE_BURO if c in NUM_PD]

    def numericas(df):
        d = df[NUM_PD].copy()
        d["buro_sin_registro"] = df[BLOQUE_BURO].isna().all(axis=1).astype(int)
        return d

    num_tr = numericas(X_train)
    medianas = num_tr[bloque_pd].median()
    num_tr = num_tr.fillna(medianas)
    medias, desv = num_tr.mean(), num_tr.std()

    def preparar(df, woe_df=None):
        num = ((numericas(df).fillna(medianas) - medias) / desv).astype("float32")
        if woe_df is None:
            woe_df = pd.DataFrame({f"{c}_woe": aplicar_woe(df[c], woes[c]) for c in CAT_PD}, index=df.index)
        return pd.concat([num, woe_df], axis=1)

    woe_tr = woe_train_fuera_de_muestra(X_train, y_train, seed) if WOE_FUERA_DE_MUESTRA else None
    return preparar(X_train, woe_tr), preparar(X_val), preparar(X_test)

X_train_pd, X_val_pd, X_test_pd = matrices_pd(X_train, X_val, X_test, y_train, SEED)

# ----------------------------- EAD -----------------------------
NUM_EAD = ["funded_amnt", "interest_rate", "loan_term_months", "issue_date_year",
           "annual_income_log", "dept_paym_income_ratio",
           "num_30+_delinq_in_2yrs", "mths_since_last_delinq", "never_delinq", "num_derogatory_pub_rec",
           "num_open_credit_lines", "tot_num_credit_lines", "total_credit_revolving_bal",
           "used_credit_share", "credit_history_months",
           "num_inq_in_6mths", "num_inq_in_7_12mths",
           "num_open_trades_in_6mths", "num_installment_acc_op_in_24mths",
           "num_rev_trades_op_in_24mths", "max_bal_owed", "num_inq"]
CAT_EAD = ["home_ownership_status", "verification_status", "loan_purpose", "addr_state",
           "initial_list_status", "disbursement_method", "issue_date_month",
           "emp_length_cat", "emp_ambito",
           "mths_since_last_installment_acc_op_cat", "bal_to_cred_lim_cat",
           "grade"]                                                             # permitido en EAD/LGD

def ead_observada(df):
    """Principal pendiente en el momento del impago: importe concedido − principal cobrado."""
    ead = (df["funded_amnt"] - df["princ_rec"]).clip(lower=0)
    return pd.DataFrame({"ead_obs": ead, "ead_pct": ead / df["funded_amnt"]}, index=df.index)

def lgd_observada(X, y):
    """EAD, recuperación, pérdida y LGD observadas. La LGD solo existe en los impagos con EAD > 0."""
    y = np.asarray(y).astype(int).ravel()
    d = X[["funded_amnt", "paym_rec", "princ_rec", "interest_rec", "late_fees_rec"]].astype(float)
    o = pd.DataFrame(index=X.index)
    o["y"] = y
    o["ead"] = (d["funded_amnt"] - d["princ_rec"]).clip(lower=0)
    o["recuperacion"] = (d["paym_rec"] - d[["princ_rec", "interest_rec", "late_fees_rec"]].sum(axis=1)).clip(lower=0)
    o.loc[o["y"] == 0, "recuperacion"] = 0.0
    o["perdida"] = o["y"] * (o["ead"] - o["recuperacion"]).clip(lower=0)
    o["lgd"] = np.where((o["y"] == 1) & (o["ead"] > 0),
                        (o["perdida"] / o["ead"].where(o["ead"] > 0)).clip(0, 1), np.nan)
    return o

malos_train = X_train.loc[y_train == 1]
CATEGORIAS_EAD = {c: sorted(malos_train[c].dropna().unique()) for c in CAT_EAD}

def matriz_ead(df):
    """Numéricas sin imputar + categóricas con las categorías de los impagos de train (no vista -> NaN)."""
    X = df[NUM_EAD].astype("float32").copy()
    for c in CAT_EAD:
        X[c] = pd.Categorical(df[c], categories=CATEGORIAS_EAD[c])
    return X

X_train_EAD = matriz_ead(malos_train)
y_train_EAD = ead_observada(malos_train)["ead_pct"]
X_val_EAD_cartera = matriz_ead(X_val)      # toda la cartera de validación (para la calibración)
X_test_EAD = matriz_ead(X_test)            # toda la cartera de test

# ----------------------------- LGD -----------------------------
lgd_obs_train = lgd_observada(X_train, y_train)
lgd_obs_val = lgd_observada(X_val, y_val)            # su pérdida real se usa en la calibración

idx_train_LGD = X_train_EAD.index[lgd_obs_train.loc[X_train_EAD.index, "ead"].to_numpy() > 0]
X_train_LGD = X_train_EAD.loc[idx_train_LGD].copy()
y_train_LGD = lgd_obs_train.loc[idx_train_LGD, "lgd"].rename("lgd")
X_val_LGD_cartera = X_val_EAD_cartera.copy()
X_test_LGD = X_test_EAD.copy()

# Dos categóricas que usa la LGD y no estaban en la matriz de EAD (categorías: impagos de train con EAD > 0)
CAT_EXTRA_LGD = ["mths_since_last_delinq_cat", "num_derogatory_pub_rec_cat"]
CATEGORIAS_EXTRA_LGD = {c: sorted(X_train.loc[idx_train_LGD, c].dropna().unique()) for c in CAT_EXTRA_LGD}
for c in CAT_EXTRA_LGD:
    X_train_LGD[c] = pd.Categorical(X_train.loc[X_train_LGD.index, c], categories=CATEGORIAS_EXTRA_LGD[c])
    X_val_LGD_cartera[c] = pd.Categorical(X_val[c], categories=CATEGORIAS_EXTRA_LGD[c])
    X_test_LGD[c] = pd.Categorical(X_test[c], categories=CATEGORIAS_EXTRA_LGD[c])

# --------------------- Comprobaciones ---------------------
assert X_train_pd.shape[1] == 22 and not X_train_pd.isna().any().any()
assert not set(X_train_pd.columns) & set(VARIABLES_DESEMPEÑO + ["grade"]), "Fuga en la matriz de PD"
assert list(X_val_pd.columns) == list(X_train_pd.columns) == list(X_test_pd.columns)
assert X_train_EAD.index.equals(y_train_EAD.index) and y_train_LGD.notna().all()
assert X_val_EAD_cartera.index.equals(X_val.index) and X_test_EAD.index.equals(X_test.index)

print(f"PD : train {X_train_pd.shape} | val {X_val_pd.shape} | test {X_test_pd.shape}")
print(f"EAD: train {X_train_EAD.shape} (impagos) | cartera val {X_val_EAD_cartera.shape} | cartera test {X_test_EAD.shape}")
print(f"LGD: train {X_train_LGD.shape} (impagos con EAD > 0; {len(X_train_EAD) - len(X_train_LGD)} descartados por EAD = 0)")
print(f"Pérdida real en validación: {lgd_obs_val['perdida'].sum():,.0f} $")

# --------------------- Limpieza de memoria ---------------------
# De test solo se guarda lo necesario para calcular la pérdida real en la celda final
COLS_PERDIDA = ["funded_amnt", "paym_rec", "princ_rec", "interest_rec", "late_fees_rec"]
X_test_perdida = X_test[COLS_PERDIDA].copy()
del X_train, X_val, X_test, malos_train

PD : train (1283785, 22) | val (427929, 22) | test (427929, 22)
EAD: train (168075, 34) (impagos) | cartera val (427929, 34) | cartera test (427929, 34)
LGD: train (168023, 36) (impagos con EAD > 0; 52 descartados por EAD = 0)
Pérdida real en validación: 554,073,730 $


#3.Entrenamiento de los tres modelos

En esta celda se entrenan los tres modelos en los que se basará la predicción final de ECL.

Observar que el modelo seleccionado para EAD y LGD es una regresión logística fracional (LGD es por naturaleza una variable con rango [0,1] y EAD se modela como EAD=EAD%*funded_amnt, donde funded_amnt es el principal concedido con el préstamo y EAD% es el porcentaje del principal expuesto en el momento de default, variable  con rango [0,1]).

Además, la clase GLMFraccional contiene la estructura necesaria para realizar el target encoding de las variables categóricas utilizadas en los modelos de EAD y LGD. En este caso sí se realizó mediante KFold desde la primera versión del proyecto en el notebook de desarrollo, con lo que se sigue la misma metodología.

In [6]:
# =====================================================================
# ENTRENAMIENTO DE LOS TRES MODELOS
# =====================================================================
import warnings
import lightgbm as lgb
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold, train_test_split

# ------------------ Regresión logística fraccional (EAD y LGD) ------------------
SUAVIZADO_TE = 100      # m: peso de la media global en categorías con pocos impagos

def con_nulos_str(s):
    return s.astype(object).where(s.notna(), "(nulo)").astype(str)

def ajustar_te(x, y, m=SUAVIZADO_TE):
    """Media suavizada de y por categoría: (n·media + m·media_global) / (n + m)."""
    media_global = y.mean()
    stats = y.groupby(x).agg(["sum", "count"])
    return (stats["sum"] + m * media_global) / (stats["count"] + m), media_global

def aplicar_te(x, mapa, media_global):
    return x.map(mapa).fillna(media_global).astype(float)        # categoría no vista -> media global


class GLMFraccional:
    """Logística fraccional (QMLE binomial) con target encoding fuera de muestra, imputación por mediana,
    indicadores de nulo, estandarización y año categórico. Todo se aprende en el X, y de fit()."""

    def __init__(self, num, cats, nulos=(), interaccion_tipo_año=False, n_bloques_te=5):
        self.num = list(num)                    # numéricas (puede incluir "funded_amnt_log")
        self.cats = list(cats)                  # categóricas que se codifican con target encoding
        self.nulos = list(nulos)                # [(nombre_indicador, columna_original), ...]
        self.interaccion = interaccion_tipo_año # EAD: interest_rate * C(año); LGD: solo C(año)
        self.n_bloques_te = n_bloques_te

    # --- piezas de la matriz ---
    def _numericas(self, X):
        d = X[[c for c in self.num if c != "funded_amnt_log"]].astype(float).copy()
        if "funded_amnt_log" in self.num:
            d["funded_amnt_log"] = np.log(X["funded_amnt"].astype(float))
        return d

    def _te_completo(self, X):
        return pd.DataFrame({f"te_{c}": aplicar_te(con_nulos_str(X[c]), *self.mapas_te[c]) for c in self.cats},
                            index=X.index)

    def _datos(self, X, te):
        num = self._numericas(X)
        d = pd.concat([num.fillna(self.medianas), te], axis=1)
        for nombre, col in self.nulos:
            d[nombre] = X[col].isna().astype(float)
        año = X["issue_date_year"].astype(int).astype(str)
        d["año"] = año.where(año.isin(self.niveles_año), self.niveles_año[0])   # año no visto -> referencia
        d[self.cols_estandar] = (d[self.cols_estandar] - self.medias) / self.desv
        return d

    # --- ajuste y predicción ---
    def fit(self, X, y, seed):
        y = pd.Series(np.asarray(y, dtype=float), index=X.index, name="target")
        cats_tr = {c: con_nulos_str(X[c]) for c in self.cats}

        # Target encoding fuera de muestra en train (cada préstamo no usa su propio target)
        te_tr = pd.DataFrame(index=X.index, columns=[f"te_{c}" for c in self.cats], dtype=float)
        for i_aj, i_ap in KFold(self.n_bloques_te, shuffle=True, random_state=seed).split(X):
            i_aj, i_ap = X.index[i_aj], X.index[i_ap]
            for c in self.cats:
                mapa, mg = ajustar_te(cats_tr[c].loc[i_aj], y.loc[i_aj])
                te_tr.loc[i_ap, f"te_{c}"] = aplicar_te(cats_tr[c].loc[i_ap], mapa, mg).to_numpy()
        # Val y test: mapa con todo train
        self.mapas_te = {c: ajustar_te(cats_tr[c], y) for c in self.cats}

        # Imputación, año de referencia (moda) y estandarización, aprendidos en train
        num_tr = self._numericas(X)
        self.medianas = num_tr.median()
        años = X["issue_date_year"].astype(int).astype(str)
        moda = años.mode()[0]
        self.niveles_año = [moda] + sorted(a for a in años.unique() if a != moda)
        self.cols_estandar = list(num_tr.columns) + list(te_tr.columns)
        base = pd.concat([num_tr.fillna(self.medianas), te_tr], axis=1)
        self.medias = base[self.cols_estandar].mean()
        self.desv = base[self.cols_estandar].std().replace(0, 1)

        datos = self._datos(X, te_tr)
        datos["target"] = y.to_numpy()

        termino_año = (f"interest_rate * C(año, levels={self.niveles_año!r})" if self.interaccion
                       else f"C(año, levels={self.niveles_año!r})")
        terminos = ([termino_año] + [f"Q('{c}')" for c in self.num if not (self.interaccion and c == "interest_rate")]
                    + [n for n, _ in self.nulos] + list(te_tr.columns))
        self.formula = "target ~ " + " + ".join(terminos)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")       # aviso esperado: target no binario (regresión fraccional)
            self.modelo = smf.glm(self.formula, data=datos, family=sm.families.Binomial()).fit(cov_type="HC0")
        assert self.modelo.converged, "La regresión fraccional no convergió"
        return self

    def predict(self, X):
        return pd.Series(self.modelo.predict(self._datos(X, self._te_completo(X))).to_numpy(), index=X.index)


# ------------------ EAD: modelo 3 (logística fraccional reducida) ------------------
NUM_EAD_M3 = ["loan_term_months", "num_open_credit_lines", "tot_num_credit_lines", "used_credit_share",
          "num_inq_in_6mths", "never_delinq", "max_bal_owed", "num_derogatory_pub_rec",
          "dept_paym_income_ratio", "credit_history_months", "num_rev_trades_op_in_24mths",
          "annual_income_log", "funded_amnt_log", "mths_since_last_delinq"]
CAT_EAD_M3 = ["bal_to_cred_lim_cat", "mths_since_last_installment_acc_op_cat", "issue_date_month",
          "emp_length_cat", "loan_purpose", "addr_state"]                  # -> TE_M3
# En el desarrollo, el indicador del bloque de buró era el patrón de nulos de num_open_trades_in_6mths
# (primera variable del bloque en el orden de NUM_EAD)
NULOS_EAD_M3 = [("nulo_bloque_buro", "num_open_trades_in_6mths")]

modelo_ead = GLMFraccional(num=["interest_rate"] + NUM_EAD_M3, cats=CAT_EAD_M3, nulos=NULOS_EAD_M3,
                           interaccion_tipo_año=True).fit(X_train_EAD, y_train_EAD, SEED)
print(f"EAD · parámetros: {len(modelo_ead.modelo.params)} | EAD% media predicha en train: "
      f"{modelo_ead.predict(X_train_EAD).mean():.4f} (real {y_train_EAD.mean():.4f})")

# ------------------ LGD: modelo 2 (logística fraccional reducida) ------------------
NUM_LGD_M2 = ["loan_term_months", "interest_rate", "annual_income_log", "num_open_credit_lines",
              "tot_num_credit_lines", "funded_amnt_log", "num_rev_trades_op_in_24mths", "max_bal_owed"]
CAT_LGD_M2 = ["addr_state", "emp_ambito", "home_ownership_status", "verification_status",
              "emp_length_cat", "num_derogatory_pub_rec_cat"]              # -> TE_LGD_M2

modelo_lgd = GLMFraccional(num=NUM_LGD_M2, cats=CAT_LGD_M2).fit(X_train_LGD, y_train_LGD, SEED)
print(f"LGD · parámetros: {len(modelo_lgd.modelo.params)} | LGD media predicha en train: "
      f"{modelo_lgd.predict(X_train_LGD).mean():.4f} (real {y_train_LGD.mean():.4f})")

# ------------------ PD:  (LightGBM+) ------------------
#PARAMS_PD = dict(
    #objective="binary", linear_tree=True, n_estimators=4000, subsample_freq=1,
    #learning_rate=0.011064856929825496, num_leaves=40, min_child_samples=433, linear_lambda=0.722345183521419,   #estos son los de la optimización del GBM+LT con WoE fuera de muestra
    #reg_lambda=0.0011047270150509982, subsample=0.9542974508089956, colsample_bytree=0.4490608665367354,
    #random_state=SEED, deterministic=True, force_row_wise=True, verbose=-1, n_jobs=-1)

PARAMS_PD = dict(
      objective="binary", linear_tree=False, n_estimators=5000, subsample_freq=1,
      learning_rate=0.015442168320935673, num_leaves=40, min_child_samples=208,
      reg_lambda=0.0010133406281001406, subsample=0.9443870282737901,
      colsample_bytree=0.41804698005714136,
      random_state=SEED, deterministic=True, force_row_wise=True, verbose=-1, n_jobs=-1)

ES_RONDAS_PD = 100

# Número de árboles: early stopping con un 10 % estratificado de train (validación no interviene)
i_fit, i_es = train_test_split(np.arange(len(y_train)), test_size=0.10, stratify=y_train, random_state=SEED)
y_tr_pd = y_train.to_numpy()
modelo_pd = lgb.LGBMClassifier(**PARAMS_PD)
modelo_pd.fit(X_train_pd.iloc[i_fit], y_tr_pd[i_fit],
              eval_set=[(X_train_pd.iloc[i_es], y_tr_pd[i_es])], eval_metric="binary_logloss",
              callbacks=[lgb.early_stopping(ES_RONDAS_PD, verbose=False)])
N_ARBOLES_PD = modelo_pd.best_iteration_
print(f"PD · árboles: {N_ARBOLES_PD}")

EAD · parámetros: 45 | EAD% media predicha en train: 0.6896 (real 0.6895)
LGD · parámetros: 26 | LGD media predicha en train: 0.9016 (real 0.9016)
PD · árboles: 2184


## Métricas

Añadimos a continuación las métricas en validación de los tres modelos por separado, a pesar de que el objetivo último es obtener una ECL calibrada lo más baja posible de manera que se cumplan a su vez las dos restricciones del reto y no el rendimiento individual de cada modelo.

Tener en cuenta que hasta ahora el conjunto de validación no ha sido utilizado en el proceso de modelización. Sin embargo, este ha servido para decidir con qué modelos quedarnos, por lo que las métricas son ligeramente optimistas.

In [7]:
# =====================================================================
# MÉTRICAS DE LOS TRES MODELOS (mismas definiciones que en el notebook de desarrollo)
# =====================================================================
from sklearn.metrics import (log_loss, roc_auc_score, brier_score_loss,
                             mean_absolute_error, mean_squared_error, r2_score)

# ----- PD (como en pd_optuna_cv.py) -----
def error_calibracion_pd(y, p, n_grupos=10):
    """Media, ponderada por tamaño, de |PD media − tasa de impago| en deciles de la PD predicha."""
    g = pd.qcut(pd.Series(p).rank(method="first"), n_grupos, labels=False)
    t = (pd.DataFrame({"g": g.to_numpy(), "y": y, "p": p})
         .groupby("g").agg(y=("y", "mean"), p=("p", "mean"), n=("y", "size")))
    return float((np.abs(t["p"] - t["y"]) * t["n"]).sum() / t["n"].sum())

def metricas_pd(y, p):
    y, p = np.asarray(y).astype(int), np.asarray(p, dtype=float)
    auc = roc_auc_score(y, p)
    return {"log_loss": log_loss(y, p), "AUC": auc, "Gini": 2 * auc - 1, "Brier": brier_score_loss(y, p),
            "error_calibración": error_calibracion_pd(y, p), "PD media": p.mean(), "tasa impago": y.mean()}

# ----- EAD -----
def metricas_ead(y_pct, pred_pct, funded):
    """Métricas de la EAD en % y en $, más calibración por deciles de la predicción."""
    y_pct = np.asarray(y_pct, dtype=float)
    pred_pct = np.clip(np.asarray(pred_pct, dtype=float), 0, 1)
    funded = np.asarray(funded, dtype=float)
    ead_real, ead_pred = y_pct * funded, pred_pct * funded
    no_cero = y_pct > 0
    calib = (pd.DataFrame({"pred": pred_pct, "real": y_pct})
             .assign(decil=lambda d: pd.qcut(d["pred"], 10, labels=False, duplicates="drop"))
             .groupby("decil").agg(pred_media=("pred", "mean"), real_media=("real", "mean")))
    return {"MAE_%": mean_absolute_error(y_pct, pred_pct),
            "RMSE_%": np.sqrt(mean_squared_error(y_pct, pred_pct)),
            "MAPE_%": np.mean(np.abs((y_pct[no_cero] - pred_pct[no_cero]) / y_pct[no_cero])),
            "R2_%": r2_score(y_pct, pred_pct),
            "MAE_$": mean_absolute_error(ead_real, ead_pred),
            "WAPE_$": np.abs(ead_real - ead_pred).sum() / ead_real.sum(),
            "EAD_total_pred/real": ead_pred.sum() / ead_real.sum(),
            "error_calibracion": (calib["pred_media"] - calib["real_media"]).abs().sum(),
            "infraestimados": (ead_pred < ead_real).mean()}

# ----- LGD -----
def metricas_lgd(y, pred, ead):
    """y: LGD observada, pred: LGD predicha, ead: EAD observada en $ (para pasar el error a pérdida)."""
    y, pred, ead = (np.asarray(a, dtype=float).ravel() for a in (y, pred, ead))
    err = pred - y
    no_cero = y > 0.01
    grupos = (pd.qcut(pd.Series(pred).rank(method="first"), 10, labels=False) if np.unique(pred).size > 1
              else pd.Series(np.zeros(len(pred))))
    cal = pd.DataFrame({"g": grupos.to_numpy(), "y": y, "p": pred}).groupby("g").agg(y=("y", "mean"), p=("p", "mean"))
    return {"MAE": np.abs(err).mean(), "RMSE": np.sqrt((err ** 2).mean()),
            "MAPE (sin LGD=0)": np.abs(err[no_cero] / y[no_cero]).mean(),
            "R²": 1 - (err ** 2).sum() / ((y - y.mean()) ** 2).sum(),
            "LGD media pred": pred.mean(), "LGD media real": y.mean(),
            "MAE pérdida ($)": np.abs(err * ead).mean(),
            "pérdida total pred / real": (pred * ead).sum() / (y * ead).sum(),
            "error calibración": np.abs(cal["p"] - cal["y"]).mean(),
            "% infraestimados": (pred < y).mean()}

En caso de querer ver las métricas, activar la opción ``compilar_metricas = True``.

In [8]:
# =====================================================================
# MÉTRICAS EN VALIDACIÓN DE LOS TRES MODELOS (comprobación frente al notebook de desarrollo)
# =====================================================================

compilar_metricas=False # compilar_metricas=True muestra las métricas de cada uno de los modelos

if compilar_metricas:
  y_val_s = pd.Series(np.asarray(y_val).astype(int), index=X_val_pd.index)

  # ----- PD: toda la cartera de validación -----
  pd_val = pd.Series(modelo_pd.predict_proba(X_val_pd, num_iteration=N_ARBOLES_PD)[:, 1], index=X_val_pd.index)
  met_pd = metricas_pd(y_val_s, pd_val)

  # ----- EAD: impagos de validación (los únicos con EAD observada) -----
  idx_val_EAD = y_val_s.index[y_val_s == 1]
  X_val_EAD = X_val_EAD_cartera.loc[idx_val_EAD]
  funded_val_EAD = X_val_EAD["funded_amnt"].astype(float)
  y_val_EAD = lgd_obs_val.loc[idx_val_EAD, "ead"] / funded_val_EAD          # EAD% observada
  met_ead = metricas_ead(y_val_EAD, modelo_ead.predict(X_val_EAD), funded_val_EAD)

  # ----- LGD: impagos de validación con EAD > 0 -----
  idx_val_LGD = idx_val_EAD[lgd_obs_val.loc[idx_val_EAD, "ead"].to_numpy() > 0]
  X_val_LGD = X_val_LGD_cartera.loc[idx_val_LGD]
  y_val_LGD = lgd_obs_val.loc[idx_val_LGD, "lgd"]
  ead_val_LGD = lgd_obs_val.loc[idx_val_LGD, "ead"]
  met_lgd = metricas_lgd(y_val_LGD, modelo_lgd.predict(X_val_LGD), ead_val_LGD)

  # ----- Resultados -----
  print(f"Validación: {len(y_val_s):,} préstamos | {len(idx_val_EAD):,} impagos | {len(idx_val_LGD):,} impagos con EAD > 0\n")
  print("PD (LightGBM estándar optimizado):")
  display(pd.Series(met_pd).to_frame("val").T.style.format("{:.5f}"))
  print("EAD (logística fraccional reducida, modelo 3):")
  display(pd.Series(met_ead).to_frame("val").T.style.format("{:.4f}").format("{:,.0f}", subset=["MAE_$"]))
  print("LGD (logística fraccional reducida, modelo 2):")
  display(pd.Series(met_lgd).to_frame("val").T.style.format("{:.4f}").format("{:,.0f}", subset=["MAE pérdida ($)"]))

else:
  None

Las métricas obtenidas son informativas del rendimiento de los modelos por separado. Sin embargo, en último término lo importante será el resultado que da la combinación de ellos junto con la calibración.

#4.Predicciones sobre la cartera de validación

A continuación se predice PD, EAD y LGD sobre la cartera de validación completa. Estas predicciones son la base para la posterior calibración.

In [9]:
# =====================================================================
# PREDICCIONES SOBRE LA CARTERA DE VALIDACIÓN (entrada de la calibración)
# La función predecir_cartera también nos servirá para el test
# =====================================================================
def predecir_cartera(X_pd, X_ead, X_lgd):
    """PD, EAD y LGD predichas para todos los préstamos de una cartera, y su ECL sin calibrar.
    Sirve igual para validación y para test (no usa ni el impago real ni las variables de desempeño)."""
    assert X_pd.index.equals(X_ead.index) and X_ead.index.equals(X_lgd.index), "Las matrices no están alineadas"
    c = pd.DataFrame(index=X_pd.index)
    c["pd"] = modelo_pd.predict_proba(X_pd, num_iteration=N_ARBOLES_PD)[:, 1]
    c["funded"] = X_ead["funded_amnt"].astype(float)
    c["ead_pct_pred"] = modelo_ead.predict(X_ead).to_numpy()
    c["lgd_pred"] = modelo_lgd.predict(X_lgd).to_numpy()
    c["ead_usd_pred"] = c["ead_pct_pred"] * c["funded"]  #Nuestro modelo para EAD predice EAD%, luego EAD=funded*EAD%
    c["ecl_pred"] = c["pd"] * c["lgd_pred"] * c["ead_usd_pred"]          # PD · LGD · EAD
    return c

cartera_val = predecir_cartera(X_val_pd, X_val_EAD_cartera, X_val_LGD_cartera)

# Impago y pérdida real de validación: solo aquí, porque la calibración aprende de ellos
cartera_val["y"] = np.asarray(y_val).astype(int)
cartera_val["perdida"] = lgd_obs_val.loc[cartera_val.index, "perdida"].to_numpy()

# ----- Comprobaciones y resumen -----
assert cartera_val.notna().all().all(), "Hay predicciones nulas en la cartera de validación"
ecl_tot, perd_tot = cartera_val["ecl_pred"].sum(), cartera_val["perdida"].sum()
print(f"Cartera de validación: {len(cartera_val):,} préstamos | impagos: {cartera_val['y'].sum():,}")
print(f"ECL sin calibrar: {ecl_tot:,.0f} $ | pérdida real: {perd_tot:,.0f} $ | cociente: {ecl_tot / perd_tot:.4f}")
print(f"Préstamos con ECL_j < pérdida_j (sin calibrar): {(cartera_val['ecl_pred'] < cartera_val['perdida']).mean():.2%}")

Cartera de validación: 427,929 préstamos | impagos: 55,666
ECL sin calibrar: 556,174,269 $ | pérdida real: 554,073,730 $ | cociente: 1.0038
Préstamos con ECL_j < pérdida_j (sin calibrar): 12.73%


# 5.Calibración

Para cumplir con las restricciones del reto
* ECL predicha > ECL real
* ECL_j predicha < ECL_j real como máximo para el 10% de la cartera de test

se hace necesario calibrar las predicciones.

A continuación, implementamos la calibración híbrida descrita en el notebook de desarrollo.

In [10]:
# =====================================================================
# CALIBRACIÓN HÍBRIDA (aprende en validación; los parámetros se congelan para test)
#   s_j = PD_j / (LGD_j · EAD_j)
#   ECL_j = min(φ · LGD_j · EAD_j, funded_j)   si s_j ≥ θ   (tramo alto)
#   ECL_j = κ · PD_j · LGD_j · EAD_j            si s_j < θ   (tramo bajo)
# =====================================================================
from scipy.stats import norm

COBERTURA = 0.90                               # límite del reto
EPS = 0.005                                    # margen sobre la cobertura
KAPPA_MIN = 1e-3                               # κ > 0 para que los no impagos queden cubiertos
CONF_TOTAL = 0.999                             # confianza de la restricción del total
B_DELTA = 100                                  # réplicas bootstrap para δ
THETA_Q = np.round(np.arange(0.70, 0.985, 0.005), 3)
PHIS = np.round(np.arange(0.80, 2.01, 0.02), 2)
Z_TOTAL = norm.ppf(CONF_TOTAL)
EJECUTAR_OOB = False                           # comprobación out-of-bag (solo diagnóstico; lenta)
B_OOB = 100

def entradas_calibracion(c):
    """Arrays que usa la calibración a partir de una cartera (val o test)."""
    le = (c["lgd_pred"] * c["ead_usd_pred"]).to_numpy()            # LGD · EAD predichas ($)
    return {"score": c["pd"].to_numpy() / le,                       # impagos esperados por dólar de suelo
            "le": le,
            "techo": c["funded"].to_numpy(),
            "eb": c["ecl_pred"].to_numpy()}                         # PD · LGD · EAD

def ecl_hibrida(score, le, techo, eb, theta, phi, kappa):
    alto = score >= theta
    return np.where(alto, np.minimum(phi * le, techo), kappa * eb)

def calibrar_hibrida(score, le, techo, eb, L, rng, n_test):
    """Aprende (θ, φ, κ, δ) con los préstamos dados (pueden tener repetidos).
    n_test: tamaño de la muestra donde se aplicará. δ cubre el error de la muestra de calibración
    (que κ absorbe) y el del test: factor √(1 + n_cal / n_test)."""
    m, L_tot = len(score), L.sum()
    factor = np.sqrt(1 + m / n_test)
    phi_req = np.where(techo > L, L / le, np.inf)           # φ mínimo que cubriría cada préstamo del tramo alto

    # Paso 1: combinaciones (θ, φ) que alcanzan la cobertura, con su ECL total si δ = 0
    cands = []
    for q in THETA_Q:
        theta = np.quantile(score, q)
        alto = score >= theta
        gratis_bajo = ((~alto) & (L == 0)).sum()
        cubiertos_alto = np.searchsorted(np.sort(phi_req[alto]), PHIS, side="left")
        cob = (gratis_bajo + cubiertos_alto) / m
        coste_alto = np.minimum(np.outer(le[alto], PHIS), techo[alto][:, None]).sum(axis=0)
        eb_bajo = eb[~alto].sum()
        kappa = np.maximum(KAPPA_MIN, (L_tot - coste_alto) / eb_bajo)
        for k in np.where(cob >= COBERTURA + EPS)[0]:
            cands.append((coste_alto[k] + kappa[k] * eb_bajo, theta, PHIS[k]))
    if not cands:
        raise ValueError("Ninguna combinación alcanza la cobertura: amplía THETA_Q o PHIS")

    # Paso 2: entre las de menor ECL (empate al 0.1 %), la de menor ECL final con δ aproximado (método delta)
    total_min = min(cd[0] for cd in cands)
    mejor = None
    for total0, theta, phi in cands:
        if total0 > total_min * 1.001:
            continue
        alto = score >= theta
        e_alto = np.where(alto, np.minimum(phi * le, techo), 0.0)
        e_bajo = np.where(alto, 0.0, eb)
        k0 = max(KAPPA_MIN, (L_tot - e_alto.sum()) / e_bajo.sum())
        e0 = e_alto + k0 * e_bajo
        R = e0.sum() / L_tot
        sd = np.sqrt(m * np.var(e0 - R * L)) / L_tot
        delta_aprox = max(0.0, Z_TOTAL * factor * sd - (R - 1))
        k_final = max(KAPPA_MIN, ((1 + delta_aprox) * L_tot - e_alto.sum()) / e_bajo.sum())
        clave = (round(e_alto.sum() + k_final * e_bajo.sum(), -5), -k_final)   # menor ECL; empate → mayor κ
        if mejor is None or clave < mejor[0]:
            mejor = (clave, theta, phi)
    _, theta, phi = mejor

    # Paso 3: δ definitivo con la desviación típica bootstrap del cociente ECL / pérdida
    alto = score >= theta
    e_alto = np.where(alto, np.minimum(phi * le, techo), 0.0)
    e_bajo = np.where(alto, 0.0, eb)
    k0 = max(KAPPA_MIN, (L_tot - e_alto.sum()) / e_bajo.sum())
    e0 = e_alto + k0 * e_bajo
    ratios = np.empty(B_DELTA)
    for b in range(B_DELTA):
        i = rng.integers(0, m, m)
        ratios[b] = e0[i].sum() / L[i].sum()
    R0 = e0.sum() / L_tot
    delta = max(0.0, Z_TOTAL * factor * ratios.std() - (R0 - 1))
    kappa = max(KAPPA_MIN, ((1 + delta) * L_tot - e_alto.sum()) / e_bajo.sum())
    return {"theta": float(theta), "phi": float(phi), "kappa": float(kappa), "delta": float(delta)}

# ===== Calibración sobre toda la validación =====
ent_val = entradas_calibracion(cartera_val)
PERDIDA_VAL = cartera_val["perdida"].to_numpy()
N_TEST = len(X_test_pd)                        # tamaño del test (se conoce sin mirar sus etiquetas)

rng = np.random.default_rng(SEED)
PARAMS_HIB = calibrar_hibrida(**ent_val, L=PERDIDA_VAL, rng=rng, n_test=N_TEST)   # congelados para test

cartera_val["ecl_hib"] = ecl_hibrida(**ent_val, **{k: PARAMS_HIB[k] for k in ("theta", "phi", "kappa")})
e = cartera_val["ecl_hib"].to_numpy()
alto = ent_val["score"] >= PARAMS_HIB["theta"]
print(f"Parámetros congelados: θ = {PARAMS_HIB['theta']:.6g} ({alto.mean():.1%} de val en el tramo alto) | "
      f"φ = {PARAMS_HIB['phi']:.2f} | κ = {PARAMS_HIB['kappa']:.4f} | δ = {PARAMS_HIB['delta']:.2%}")
print(f"Validación · ECL = {e.sum():,.0f} $ | pérdida = {PERDIDA_VAL.sum():,.0f} $ | "
      f"ECL / pérdida = {e.sum() / PERDIDA_VAL.sum():.4f} | cubiertos = {(e > PERDIDA_VAL).mean():.2%}")
print(f"MAE de ECL_j · sin calibrar: {np.abs(ent_val['eb'] - PERDIDA_VAL).mean():,.0f} $ | "
      f"calibrada: {np.abs(e - PERDIDA_VAL).mean():,.0f} $")

# ===== Comprobación out-of-bag (opcional) =====
if EJECUTAR_OOB:
    todos = np.arange(len(cartera_val))
    filas = []
    for b in range(B_OOB):
        i_in = rng.integers(0, len(todos), len(todos))          # muestra bootstrap: aquí se calibra
        oob = np.setdiff1d(todos, i_in)                         # préstamos no vistos: aquí se evalúa
        p = calibrar_hibrida(**{k: v[i_in] for k, v in ent_val.items()}, L=PERDIDA_VAL[i_in],
                             rng=rng, n_test=len(oob))
        e_oob = ecl_hibrida(**{k: v[oob] for k, v in ent_val.items()},
                            **{k: p[k] for k in ("theta", "phi", "kappa")})
        filas.append({**p, "ratio_oob": e_oob.sum() / PERDIDA_VAL[oob].sum(),
                      "cobertura_oob": (e_oob > PERDIDA_VAL[oob]).mean()})
    oob_res = pd.DataFrame(filas)
    ok_total, ok_cob = oob_res["ratio_oob"] > 1, oob_res["cobertura_oob"] >= COBERTURA
    print(f"\nOOB ({B_OOB} réplicas) · cumple total: {ok_total.mean():.0%} (mín {oob_res['ratio_oob'].min():.4f}) | "
          f"cumple cobertura: {ok_cob.mean():.0%} (mín {oob_res['cobertura_oob'].min():.2%}) | "
          f"ambas: {(ok_total & ok_cob).mean():.0%}")

Parámetros congelados: θ = 3.74057e-05 (17.5% de val en el tramo alto) | φ = 1.54 | κ = 0.2969 | δ = 2.29%
Validación · ECL = 566,779,571 $ | pérdida = 554,073,730 $ | ECL / pérdida = 1.0229 | cubiertos = 90.52%
MAE de ECL_j · sin calibrar: 1,997 $ | calibrada: 2,127 $


#6.Predicción final sobre test

In [11]:
# =====================================================================
# PREDICCIÓN SOBRE TEST Y ECL FINAL POR PRÉSTAMO
#   Modelos entrenados en train + parámetros de calibración congelados (aprendidos en validación).
#   No se usa y_test ni las variables de desempeño de test.
# =====================================================================
cartera_test = predecir_cartera(X_test_pd, X_test_EAD, X_test_LGD)
assert cartera_test.notna().all().all(), "Hay predicciones nulas en la cartera de test"

ent_test = entradas_calibracion(cartera_test)
cartera_test["tramo"] = np.where(ent_test["score"] >= PARAMS_HIB["theta"], "alto", "bajo")
cartera_test["ecl_final"] = ecl_hibrida(**ent_test, **{k: PARAMS_HIB[k] for k in ("theta", "phi", "kappa")})

# ECL final de cada préstamo de test (con las predicciones que la componen)
ECL_TEST = cartera_test[["pd", "lgd_pred", "ead_usd_pred", "funded", "ecl_pred", "tramo", "ecl_final"]]

# ----- Resumen (sin información real de test) -----
print(f"Cartera de test: {len(ECL_TEST):,} préstamos")
print(f"Parámetros aplicados: θ = {PARAMS_HIB['theta']:.6g} | φ = {PARAMS_HIB['phi']:.2f} | "
      f"κ = {PARAMS_HIB['kappa']:.4f} (δ = {PARAMS_HIB['delta']:.2%})")
print(f"Tramo alto: {(ECL_TEST['tramo'] == 'alto').mean():.1%} de los préstamos "
      f"(en validación: {(ent_val['score'] >= PARAMS_HIB['theta']).mean():.1%})")
print(f"ECL sin calibrar: {ECL_TEST['ecl_pred'].sum():,.0f} $ | ECL final: {ECL_TEST['ecl_final'].sum():,.0f} $")

Cartera de test: 427,929 préstamos
Parámetros aplicados: θ = 3.74057e-05 | φ = 1.54 | κ = 0.2969 (δ = 2.29%)
Tramo alto: 17.6% de los préstamos (en validación: 17.5%)
ECL sin calibrar: 556,159,355 $ | ECL final: 566,958,326 $


#7.Resultados finales sobre test

In [12]:
# =====================================================================
# RESULTADOS FINALES SOBRE TEST
#   Única celda que usa y_test y las variables de desempeño de test (para la pérdida real).
#   Pérdida real de cada préstamo ("ECL_j real"): L_j = y_j · max(EAD_j − recuperación_j, 0)
# =====================================================================
perdida_test = lgd_observada(X_test_perdida, y_test).loc[ECL_TEST.index, "perdida"]

ecl_j = ECL_TEST["ecl_final"].to_numpy()
real_j = perdida_test.to_numpy()

ecl_total, real_total = ecl_j.sum(), real_j.sum()
cumple_total = ecl_total > real_total
pct_infra = (ecl_j < real_j).mean()
cumple_cobertura = pct_infra <= 0.10
mae = np.abs(ecl_j - real_j).mean()
con_perdida = real_j > 0
mape = np.abs((ecl_j[con_perdida] - real_j[con_perdida]) / real_j[con_perdida]).mean()

print(f"SEED = {SEED} | préstamos de test: {len(ecl_j):,}\n")
print(f"ECL total predicha:  {ecl_total:,.0f} $")
print(f"ECL total real:      {real_total:,.0f} $")
print(f"Restricción 1 · ECL predicha > ECL real: {'CUMPLE' if cumple_total else 'NO CUMPLE'} "
      f"(cociente {ecl_total / real_total:.4f})")
print(f"Restricción 2 · préstamos con ECL_j < ECL_j real: {pct_infra:.2%} (máximo 10 %) → "
      f"{'CUMPLE' if cumple_cobertura else 'NO CUMPLE'}")
print(f"MAE de ECL_j:  {mae:,.2f} $")
print(f"MAPE de ECL_j: {mape:.2%} (sobre los {con_perdida.sum():,} préstamos con ECL_j real > 0; "
      f"no está definido donde la ECL real es 0)")

SEED = 42 | préstamos de test: 427,929

ECL total predicha:  566,958,326 $
ECL total real:      556,332,939 $
Restricción 1 · ECL predicha > ECL real: CUMPLE (cociente 1.0191)
Restricción 2 · préstamos con ECL_j < ECL_j real: 9.50% (máximo 10 %) → CUMPLE
MAE de ECL_j:  2,128.35 $
MAPE de ECL_j: 155.20% (sobre los 55,750 préstamos con ECL_j real > 0; no está definido donde la ECL real es 0)


El MAPE de los ECL$_j$ es alto, como cabe esperar con esta calibración: en el tramo alto, la ECL de los impagos se fija en $\varphi \cdot \widehat{LGD} \cdot \widehat{EAD}$ con $\varphi > 1$, y en el tramo bajo, su ECL ($\kappa \cdot PD \cdot LGD \cdot EAD$) queda muy por debajo de su pérdida. La calibración no busca minimizar el error préstamo a préstamo, sino la ECL total sujeta a las dos restricciones.